In [38]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv('telco_churn_data.csv')
df
df.TotalCharges = df.TotalCharges.str.replace(' ', '0').astype(float)
df.drop('customerID', axis=1, inplace=True)

In [96]:
df.isna().sum()
df.duplicated().sum()


y = df['Churn']
df_object = df.select_dtypes(include='object').drop('Churn',axis=1).astype('category')
df_numeric = df.select_dtypes(include='number')

df_object.nunique()
df[df['TotalCharges'] == ' ']
df[df['tenure'] == 0]

gt3_features = df_object.nunique()[df_object.nunique() >= 3].index.tolist()
df_object_gt3 = df_object[gt3_features]
df_object_lt3 = df_object.replace({"Yes": 1, "No": 0}).replace({"Male": 1, "Female": 0}).drop(gt3_features, axis=1)

df_numeric[df_numeric['tenure'] == 0]




/tmp/ipykernel_230/2887155890.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_object_lt3 = df_object.replace({"Yes": 1, "No": 0}).replace({"Male": 1, "Female": 0}).drop(gt3_features, axis=1)
/tmp/ipykernel_230/2887155890.py:15: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_object_lt3 = df_object.replace({"Yes": 1, "No": 0}).replace({"Male": 1, "Female": 0}).drop(gt3_features, axis=1)
/tmp/ipykernel_230/2887155890.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the 

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
488,0,0,52.55,0.0
753,0,0,20.25,0.0
936,0,0,80.85,0.0
1082,0,0,25.75,0.0
1340,0,0,56.05,0.0
3331,0,0,19.85,0.0
3826,0,0,25.35,0.0
4380,0,0,20.00,0.0
5218,0,0,19.70,0.0
6670,0,0,73.35,0.0


In [75]:
from sklearn.preprocessing import OneHotEncoder

def encoder_ohe(df, features):

  ohe = OneHotEncoder()

  for f in features:
    encoded_f = ohe.fit_transform(df[[f]]).toarray()
    encoded_df = pd.DataFrame(encoded_f, columns=[f"{cat}_encoded" for cat in ohe.categories_[0]])
    df = pd.concat([df, encoded_df],axis=1).drop(f, axis=1)

  return df

df_encoded = encoder_ohe(df_object_gt3[gt3_features], gt3_features)
df_object[gt3_features].head(50)
df_encoded



,No_encoded,No phone service_encoded,Yes_encoded,DSL_encoded,Fiber optic_encoded,No_encoded,No_encoded,No internet service_encoded,Yes_encoded,No_encoded,...,No_encoded,No internet service_encoded,Yes_encoded,Month-to-month_encoded,One year_encoded,Two year_encoded,Bank transfer (automatic)_encoded,Credit card (automatic)_encoded,Electronic check_encoded,Mailed check_encoded
0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7039,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
7040,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
7041,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [81]:
df_encoded.info()
y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   No_encoded                         7043 non-null   float64
 1   No phone service_encoded           7043 non-null   float64
 2   Yes_encoded                        7043 non-null   float64
 3   DSL_encoded                        7043 non-null   float64
 4   Fiber optic_encoded                7043 non-null   float64
 5   No_encoded                         7043 non-null   float64
 6   No_encoded                         7043 non-null   float64
 7   No internet service_encoded        7043 non-null   float64
 8   Yes_encoded                        7043 non-null   float64
 9   No_encoded                         7043 non-null   float64
 10  No internet service_encoded        7043 non-null   float64
 11  Yes_encoded                        7043 non-null   float

In [102]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

X_train, X_test, y_train, y_test = train_test_split(pd.concat([df_numeric, df_encoded], axis=1), y, test_size=0.2, random_state=42)

dtc = DecisionTreeClassifier(max_depth=7)
rfc = RandomForestClassifier(max_depth=10, n_estimators=100, min_samples_split=3, random_state=42)
gbc = GradientBoostingClassifier(max_depth=7, n_estimators=1000, loss='exponential', learning_rate=0.05)

dtc.fit(X_train, y_train)
rfc.fit(X_train, y_train)
gbc.fit(X_train, y_train)

y_pred_dtc = dtc.predict(X_test)
y_pred_rfc = rfc.predict(X_test)
y_pred_gbc = gbc.predict(X_test)

print(f"DecisionTreeClassifier Accuracy: {accuracy_score(y_test, y_pred_dtc)}")
print(f"RandomForestClassifier Accuracy: {accuracy_score(y_test, y_pred_rfc)}")
print(f"GradientBoostingClassifier Accuracy: {accuracy_score(y_test, y_pred_gbc)}")





DecisionTreeClassifier Accuracy: 0.794889992902768
RandomForestClassifier Accuracy: 0.8097941802696949
GradientBoostingClassifier Accuracy: 0.7863733144073811


In [103]:
joblib.dump(rfc, "rfc_best_model.pkl")

['rfc_best_model.pkl']